In [1]:
!pip install torch torchvision scikit-learn matplotlib pillow

In [2]:
from google.colab import files
uploaded = files.upload()

Saving Training.zip to Training.zip


In [3]:
!unzip Training.zip

Archive:  Training.zip
   creating: Training/
  inflating: Training/notsmoking_0001.jpg  
  inflating: Training/notsmoking_0002.jpg  
  inflating: Training/notsmoking_0003.jpg  
  inflating: Training/notsmoking_0005.jpg  
  inflating: Training/notsmoking_0007.jpg  
  inflating: Training/notsmoking_0008.jpg  
  inflating: Training/notsmoking_0009.jpg  
  inflating: Training/notsmoking_0010.jpg  
  inflating: Training/notsmoking_0011.jpg  
  inflating: Training/notsmoking_0012.jpg  
  inflating: Training/notsmoking_0013.jpg  
  inflating: Training/notsmoking_0015.jpg  
  inflating: Training/notsmoking_0016.jpg  
  inflating: Training/notsmoking_0019.jpg  
  inflating: Training/notsmoking_0020.jpg  
  inflating: Training/notsmoking_0021.jpg  
  inflating: Training/notsmoking_0022.jpg  
  inflating: Training/notsmoking_0023.jpg  
  inflating: Training/notsmoking_0025.jpg  
  inflating: Training/notsmoking_0027.jpg  
  inflating: Training/notsmoking_0028.jpg  
  inflating: Training/notsmoki

In [4]:
import os
import shutil

os.makedirs("DATA_SET/Smokers", exist_ok=True)
os.makedirs("DATA_SET/Non-Smoker", exist_ok=True)

source_folder = "Training"

for filename in os.listdir(source_folder):
    src_path = os.path.join(source_folder, filename)

    if filename.startswith("smoking"):
        shutil.copy(src_path, "DATA_SET/Smokers")
    elif filename.startswith("notsmoking"):
        shutil.copy(src_path, "DATA_SET/Non-Smoker")

print("Dataset reorganized successfully.")

Dataset reorganized successfully.


In [5]:
!ls DATA_SET

Non-Smoker  Smokers


In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import classification_report

In [7]:
DATA_DIR = "DATA_SET"
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 10

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

Using device: cpu


In [8]:
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [9]:
dataset = datasets.ImageFolder(DATA_DIR, transform=transform)

train_size = int(0.8 * len(dataset))
val_size = int(0.1 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_data, val_data, test_data = random_split(
    dataset, [train_size, val_size, test_size]
)

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_data, batch_size=1)

class_names = dataset.classes

print("Classes:", class_names)
print("Total images:", len(dataset))

Classes: ['Non-Smoker', 'Smokers']
Total images: 716


In [10]:
model = models.efficientnet_b0(weights="DEFAULT")
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 1)

model = model.to(DEVICE)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

print("Model loaded successfully.")

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 122MB/s] 


Model loaded successfully.


In [11]:
def train_model():
    model.train()

    for epoch in range(EPOCHS):
        running_loss = 0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images = images.to(DEVICE)
            labels = labels.float().to(DEVICE)

            optimizer.zero_grad()
            outputs = model(images).squeeze()
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            preds = (torch.sigmoid(outputs) > 0.5).float()

            correct += (preds == labels).sum().item()
            total += labels.size(0)

        acc = correct / total
        print(f"Epoch [{epoch+1}/{EPOCHS}] Loss: {running_loss:.4f} Accuracy: {acc:.4f}")

In [12]:
def evaluate():
    model.eval()
    y_true = []
    y_pred = []

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(DEVICE)
            outputs = model(images).squeeze()
            preds = (torch.sigmoid(outputs) > 0.5).float()

            y_true.append(labels.item())
            y_pred.append(preds.item())

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=class_names))

In [13]:
train_model()

Epoch [1/10] Loss: 11.7345 Accuracy: 0.6486
Epoch [2/10] Loss: 9.2724 Accuracy: 0.8304
Epoch [3/10] Loss: 6.9273 Accuracy: 0.8776
Epoch [4/10] Loss: 4.8468 Accuracy: 0.9108
Epoch [5/10] Loss: 3.7012 Accuracy: 0.9283
Epoch [6/10] Loss: 2.8749 Accuracy: 0.9441
Epoch [7/10] Loss: 1.8464 Accuracy: 0.9755
Epoch [8/10] Loss: 1.3062 Accuracy: 0.9913
Epoch [9/10] Loss: 1.0220 Accuracy: 0.9878
Epoch [10/10] Loss: 0.8792 Accuracy: 0.9913


In [14]:
evaluate()


Classification Report:
              precision    recall  f1-score   support

  Non-Smoker       0.89      0.84      0.86        37
     Smokers       0.84      0.89      0.86        36

    accuracy                           0.86        73
   macro avg       0.86      0.86      0.86        73
weighted avg       0.86      0.86      0.86        73



In [15]:
torch.save(model.state_dict(), "smoker_classifier.pth")
print("Model saved successfully.")

Model saved successfully.
